# آزمایشگاه فصل پنجم: الگوهای پرتکرار و قواعد انجمنی

داده‌ها **کاملاً ساختگی** هستند. این Notebook روی هشت سبد مثال و کد همراه مستقل است؛ به شبکه نیاز ندارد. همهٔ فرمول‌ها و کدها انگلیسی‌اند و توضیح فارسی بیرون فرمول قرار می‌گیرد.

In [1]:
from pathlib import Path
from chapter05_examples import *
path = Path('baskets_synthetic.csv')
baskets=read_baskets(path)
print('Number of baskets:',len(baskets))
for i, basket in enumerate(baskets,1):print(f'T{i}:', ', '.join(sorted(basket)))

Number of baskets: 8
T1: bread, eggs, milk
T2: bread, milk
T3: beer, bread, diaper, milk
T4: beer, bread, diaper
T5: beer, diaper, milk
T6: bread, diaper, milk
T7: bread, eggs
T8: eggs, milk


## ۱. شمارش پشتیبانی

پشتیبانی یک مجموعه به معنی تعداد سبدهای حاوی تمام اقلام آن است، نه مجموع تعداد اقلام خریداری‌شده.

In [2]:
for item in ['bread','milk','diaper','beer','eggs']:
    print(item,support_count(baskets,{item}))
print('bread + milk count =',support_count(baskets,{'bread','milk'}))
print('diaper + beer count =',support_count(baskets,{'diaper','beer'}))

bread 6
milk 6
diaper 4
beer 3
eggs 3
bread + milk count = 4
diaper + beer count = 3


## ۲. سه روش کاوش؛ خروجی واحد

Apriori با حذف نامزدها، Eclat با اشتراک شناسه‌ها و FP-Growth با درخت پیشوندی جلو می‌روند؛ پشتیبانی خروجی باید دقیقاً برابر باشد.

In [3]:
min_count=3
results={'Apriori':apriori(baskets,min_count),'Eclat':eclat(baskets,min_count),'FP-Growth':fp_growth(baskets,min_count)}
assert results['Apriori']==results['Eclat']==results['FP-Growth']
for pattern,count in sorted(results['Apriori'].items(),key=lambda x:(len(x[0]),sorted(x[0]))):
    print('+'.join(sorted(pattern)),count)

beer 3
bread 6
diaper 4
eggs 3
milk 6
beer+diaper 3
bread+diaper 3
bread+milk 4
diaper+milk 3


## ۳. قواعد و معنی اطمینان

اطمینان را با نرخ پایهٔ پیامد مقایسه کنید؛ Lift برابر یک یعنی قاعده به خودی خود نرخ پیامد را افزایش نداده است.

In [4]:
freq=results['Apriori']
rules=association_rules(freq,len(baskets),.70)
for r in rules:
    print('+'.join(r['antecedent']),'->','+'.join(r['consequent']),
          f"support={r['support']:.3f} conf={r['confidence']:.3f} lift={r['lift']:.3f}")
assert len(rules)==4

beer -> diaper support=0.375 conf=1.000 lift=2.000
diaper -> beer support=0.375 conf=0.750 lift=2.000
diaper -> bread support=0.375 conf=0.750 lift=1.000
diaper -> milk support=0.375 conf=0.750 lift=1.000


## ۴. محاسبهٔ موردی Lift و Leverage

In [5]:
for lhs,rhs in [(['diaper'],['bread']),(['diaper'],['beer']),(['beer'],['diaper'])]:
    print(lhs,'->',rhs,rule_metrics(baskets,lhs,rhs))

['diaper'] -> ['bread'] {'n': 8, 'n_A': 4, 'n_B': 6, 'n_AB': 3, 'support': 0.375, 'confidence': 0.75, 'lift': 1.0, 'leverage': 0.0}
['diaper'] -> ['beer'] {'n': 8, 'n_A': 4, 'n_B': 3, 'n_AB': 3, 'support': 0.375, 'confidence': 0.75, 'lift': 2.0, 'leverage': 0.1875}
['beer'] -> ['diaper'] {'n': 8, 'n_A': 3, 'n_B': 4, 'n_AB': 3, 'support': 0.375, 'confidence': 1.0, 'lift': 2.0, 'leverage': 0.1875}


## ۵. الگوهای بسته و بیشینه

In [6]:
closed=closed_itemsets(freq); maximal=maximal_itemsets(freq)
print('Frequent:',len(freq),'Closed:',len(closed),'Maximal:',len(maximal))
assert len(freq)==9 and len(closed)==8 and len(maximal)==5

Frequent: 9 Closed: 8 Maximal: 5


## ۶. آزمایش حساسیت آستانه

In [7]:
for m in range(1,9):
    found=apriori(baskets,m)
    assert found==eclat(baskets,m)==fp_growth(baskets,m)
    print(f'min_count={m}: {len(found)} frequent itemsets')

min_count=1: 19 frequent itemsets
min_count=2: 16 frequent itemsets
min_count=3: 9 frequent itemsets
min_count=4: 4 frequent itemsets
min_count=5: 2 frequent itemsets
min_count=6: 2 frequent itemsets
min_count=7: 0 frequent itemsets
min_count=8: 0 frequent itemsets


## ۷. جدول توافق و آزمون فوق‌هندسی

In [8]:
import math
p=math.comb(4,3)*math.comb(4,0)/math.comb(8,3)
print('one-sided conditional probability=',p)
assert abs(p-1/14)<1e-12

one-sided conditional probability= 0.07142857142857142


## ۸. پارادوکس سیمپسون: دادهٔ ساختگی

در دو گروه احتمال B در دارندگان A کمتر است، اما در کل برعکس می‌شود.

In [9]:
segments={'high':{'A_B':72,'A_notB':8,'notA_B':19,'notA_notB':1},
          'low':{'A_B':2,'A_notB':18,'notA_B':16,'notA_notB':64}}
for name,q in segments.items():
    pa=q['A_B']/(q['A_B']+q['A_notB'])
    pb=q['notA_B']/(q['notA_B']+q['notA_notB'])
    assert pa<pb
    print(name,pa,pb)
A_B=sum(q['A_B'] for q in segments.values()); A=sum(q['A_B']+q['A_notB'] for q in segments.values())
notA_B=sum(q['notA_B'] for q in segments.values()); notA=sum(q['notA_B']+q['notA_notB'] for q in segments.values())
print('pooled:',A_B/A,notA_B/notA)
assert A_B/A>notA_B/notA

high 0.9 0.95
low 0.1 0.2
pooled: 0.74 0.35


## ۹. بررسی کامل خروجی و جمع‌بندی

اگر دادهٔ ورودی دارای شناسهٔ تکراری باشد، خواندن آن باید متوقف شود. سه الگوریتم روی همین دادهٔ سالم به پاسخ واحد رسیده‌اند.

In [10]:
summary={'n':len(baskets),'frequent':len(freq),'closed':len(closed),'maximal':len(maximal),'rules':len(rules)}
print(summary)
assert summary=={'n':8,'frequent':9,'closed':8,'maximal':5,'rules':4}

{'n': 8, 'frequent': 9, 'closed': 8, 'maximal': 5, 'rules': 4}


**تمرین:** آستانهٔ شمارشی را از ۳ به ۲ تغییر دهید و روی همان مجموعه‌اقلام، قواعد سه‌قلمی را بررسی کنید. سپس نتیجه را با حداقل Confidence متفاوت مقایسه کنید. نه الگوریتم و نه این Notebook فرض نمی‌کنند که هم‌وقوعی علت خرید است.